In [1]:
import torch
import torch.nn as nn
import torch.backends.cudnn as cudnn
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt

from scipy.special import logit, expit
from sklearn.metrics import roc_curve, roc_auc_score

import pandas as pd


import random
import yaml
from pathlib import Path
import time
import datetime
from string import Template
import os
import json
#from ..dataset.abstract_dataset import DeepfakeAbstractBaseDataset

from datasets.abstract_dataset import DeepfakeAbstractBaseDataset
from detectors import DETECTOR


#from metrics.utils import get_test_metrics

In [ ]:
def load_config(path):
    with open(path, 'r') as f:
        return yaml.safe_load(f)
    
def init_seed(config):
    if config['manualSeed'] is None:
        config['manualSeed'] = random.randint(1, 10000)
    random.seed(config['manualSeed'])
    torch.manual_seed(config['manualSeed'])
    if config['cuda']:
        torch.cuda.manual_seed_all(config['manualSeed'])

In [3]:
# Adapted from test.py

def prepare_testing_data(config):
    def get_test_data_loader(config, test_name):
        # update the config dictionary with the specific testing dataset
        config = config.copy()  # create a copy of config to avoid altering the original one
        config['test_dataset'] = test_name  # specify the current test dataset
        test_set = DeepfakeAbstractBaseDataset(
                config=config,
                mode='test', 
            )
        
        test_data_loader = \
            torch.utils.data.DataLoader(
                dataset=test_set, 
                batch_size=config['test_batchSize'],
                shuffle=False, 
                num_workers=int(config['workers']),
                collate_fn=test_set.collate_fn,
                drop_last=False
            )
        return test_data_loader

    test_data_loaders = {}
    for one_test_name in config['test_dataset']:
        test_data_loaders[one_test_name] = get_test_data_loader(config, one_test_name)
    return test_data_loaders

In [4]:
@torch.no_grad()
def inference(model, data_dict):
    predictions = model(data_dict, inference=True)
    return predictions

def test_one_dataset(model, data_loader, device):
    prediction_lists = []
    #feature_lists = []
    label_lists = []
    for i, data_dict in tqdm(enumerate(data_loader), total=len(data_loader)):
        # get data
        data, label, mask, landmark = \
        data_dict['image'], data_dict['label'], data_dict['mask'], data_dict['landmark']
        label = torch.where(data_dict['label'] != 0, 1, 0)
        # move data to GPU
        data_dict['image'], data_dict['label'] = data.to(device), label.to(device)
        if mask is not None:
            data_dict['mask'] = mask.to(device)
        if landmark is not None:
            data_dict['landmark'] = landmark.to(device)

        # model forward without considering gradient computation
        predictions = inference(model, data_dict)
        label_lists += list(data_dict['label'].cpu().detach().numpy())

        # if type(model).__name__ == "UCFDetector":
        #     prediction_lists += predictions['prob']
        # else:
        #     
        prediction_lists += list(predictions['prob'].cpu().detach().numpy())
        #feature_lists += list(predictions['feat'].cpu().detach().numpy())
    
    return np.array(prediction_lists), np.array(label_lists)#np.array(feature_lists)

In [5]:
def test_epoch(model, test_data_loaders, device):

    # set model to eval mode
    model.eval()

    # define test recorder
    metrics_all_datasets = {}
    preds_labels_paths = {}


    # testing for all test data
    keys = test_data_loaders.keys()
    for key in keys:
        data_dict = test_data_loaders[key].dataset.data_dict
        # compute loss for each dataset
        predictions_nps, label_nps = test_one_dataset(model, test_data_loaders[key], device)

        return (predictions_nps, label_nps, data_dict['image'])
        
        # compute metric for each dataset
        metric_one_dataset, preds_labels_paths_one_dataset = get_test_metrics(y_pred=predictions_nps, y_true=label_nps,
                                              img_names=data_dict['image'])
        metrics_all_datasets[key] = metric_one_dataset
        preds_labels_paths[key] = preds_labels_paths_one_dataset
        
        # info for each dataset
        tqdm.write(f"dataset: {key}")
        for k, v in metric_one_dataset.items():

            if k == "pred" or k == "label" or k == "paths":
                continue

            tqdm.write(f"{k}: {v}")

    #return metrics_all_datasets, preds_labels_paths

In [ ]:
def init(config_path, weights_path, test_dataset, config_test):
        config = load_config(config_path)
        test_config = load_config(config_test)

        config.update(test_config)
        config["test_dataset"] = test_dataset
        config["weights_path"] = weights_path

        if config['cudnn']:
                cudnn.benchmark = True

        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        init_seed(config)


        test_data_loaders = prepare_testing_data(config)

        model_class = DETECTOR[config['model_name']]
        model = model_class(config).to(device)

        ckpt = torch.load(config["weights_path"], map_location=device)

        if 'state_dict' in ckpt:
                ckpt = ckpt['state_dict']

        new_weights = {}

        for key, value in ckpt.items():
                new_key = key.replace('module.', '')
                if 'base_model.' in new_key:
                        new_key = new_key.replace('base_model.', 'backbone.')
                if 'classifier.' in new_key:
                        new_key = new_key.replace('classifier.', 'head.')
                if 'HRNet_layer.' in new_key:
                        new_key = new_key.replace('HRNet_layer.', 'backbone.')
                new_weights[new_key] = value

        if type(model).__name__ == "EffortDetector":
                model.load_state_dict(new_weights, strict=False)
        else:
                model.load_state_dict(new_weights, strict=True)
        print('===> Load checkpoint done!')

        return test_data_loaders, model, device, config


In [7]:
def get_prediction_df(predictions_nps, labels_nps, images):

    videos_real = set([p.split('/')[9] for p in images if p.split('/')[7] == 'real'])
    videos_fake = set([p.split('/')[9] for p in images if p.split('/')[7] == 'fake'])

    all_videos = videos_fake.union(videos_real)

    assert(len(videos_real) == 42)
    assert(len(videos_fake) == 42)

    df_by_frame = pd.DataFrame(images, columns=["video_path"])
    df_by_frame["video_name"] = [p.split('/')[9] for p in images]
    df_by_frame["pred"] = predictions_nps
    df_by_frame["label"] = labels_nps

    df_video_preds = pd.DataFrame(all_videos, columns=["video_name"])
    df_video_preds["label"] = df_video_preds.apply(lambda row: 1 if row["video_name"] in videos_fake else 0, axis=1)


    for video_name in all_videos:
        video_frames = df_by_frame[df_by_frame["video_name"] == video_name]

        #preds_sum = video_frames["preds"].sum()

        preds_clipped = np.clip(video_frames["pred"], 1e-7, 1 - 1e-7)
        preds_logit = logit(preds_clipped)
        avg_logit = np.mean(preds_logit)
        prediction_class_log = 0 if expit(avg_logit) < 0.5 else 1

        preds_mean = video_frames['pred'].mean()
        prediction_class_mean = 0 if preds_mean < 0.5 else 1

        if prediction_class_log != prediction_class_mean:
            print("mismatch", f"pred log {expit(avg_logit)}, pred normal {preds_mean}, class log {prediction_class_log}, class normal {prediction_class_mean}, vid {video_name}")

        df_video_preds.loc[df_video_preds["video_name"] == video_name, "class_log"] = prediction_class_log
        df_video_preds.loc[df_video_preds["video_name"] == video_name, "class_mean"] = prediction_class_mean
        df_video_preds.loc[df_video_preds["video_name"] == video_name, "raw_pred_log"] = expit(avg_logit)
        df_video_preds.loc[df_video_preds["video_name"] == video_name, "raw_pred_mean"] = preds_mean
        df_video_preds.loc[df_video_preds["video_name"] == video_name, "max_pred"] = video_frames['pred'].max()

    return df_video_preds

In [ ]:
def get_prediction_df_video_level(predictions_nps, labels_nps, images):

    preds_by_video = {}
    all_videos = []
    label_by_video = {}

    for index, image in enumerate(images):
        video_name = image[0].split('/')[9]
        
        if video_name not in label_by_video.keys():
            label_by_video[video_name] = labels_nps[index]
        all_videos.append(video_name)

        if video_name in preds_by_video.keys():
            preds_by_video[video_name] = [*preds_by_video[video_name], *predictions_nps[index]]
        else:
            preds_by_video[video_name] = predictions_nps[index]


    all_videos = set(all_videos)

    assert(len(all_videos) == 84)

    df_video_preds = pd.DataFrame(all_videos, columns=["video_name"])

    df_video_preds["label"] = df_video_preds.iloc[:, 0].map(label_by_video)


    for video_name in all_videos:

        preds_clipped = np.clip(preds_by_video[video_name], 1e-7, 1 - 1e-7)
        preds_logit = logit(preds_clipped)
        avg_logit = np.mean(preds_logit)
        prediction_class_log = 0 if expit(avg_logit) < 0.5 else 1

        df_preds = pd.DataFrame(preds_by_video[video_name])
        
        preds_mean = df_preds.mean()

        prediction_class_mean = 0 if preds_mean.squeeze() < 0.5 else 1

        if prediction_class_log != prediction_class_mean:
            print("mismatch", f"pred log {expit(avg_logit)}, pred normal {preds_mean}, class log {prediction_class_log}, class normal {prediction_class_mean}, vid {video_name}")

        df_video_preds.loc[df_video_preds["video_name"] == video_name, "class_log"] = prediction_class_log
        df_video_preds.loc[df_video_preds["video_name"] == video_name, "class_mean"] = prediction_class_mean
        df_video_preds.loc[df_video_preds["video_name"] == video_name, "raw_pred_log"] = expit(avg_logit)
        df_video_preds.loc[df_video_preds["video_name"] == video_name, "raw_pred_mean"] = preds_mean
        df_video_preds.loc[df_video_preds["video_name"] == video_name, "max_pred"] = max(preds_by_video[video_name])


    return df_video_preds

In [9]:
class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        if isinstance(obj, np.float32):
            return obj.item()
        if isinstance(obj, np.int64):
            return obj.item()
        return super().default(obj)

In [ ]:
def save_display_metrics(predictions, labels, images, model_name, test_dataset, weights_path, elapsed_time, config, exps_dir):
    assert(len(predictions) == len(labels))
    assert(len(images) == len(labels))

    train_dataset = str(Path(weights_path).parent)
    print(f"Results for {model_name} ({train_dataset}) on {test_dataset} ({elapsed_time}s)")

    vid_level_models = ["i3d"]

    #Frame-level AUC

    if model_name in vid_level_models:
        clip_labels = []
        clip_preds = []

        vid_preds_mean = []
        vids_mean = {}
        for index, array in enumerate(predictions):
            frame_path = Path(images[index][0])
            video_path = str(frame_path.parent)
            for i in range(0, len(array)):
                clip_labels.append(labels[index])
            for pred in array:
                clip_preds.append(pred)

            vid_mean = np.mean(array)
            vid_preds_mean.append(vid_mean)

            vids_mean[video_path] = {
                "pred": vid_mean,
                "label": labels[index]
            }

        fpr, tpr, thresholds = roc_curve(clip_labels, clip_preds)
        auc_score = roc_auc_score(clip_labels, clip_preds)
        print(f"AUC (frame-level): {auc_score}")

        #Video-level AUC
        fpr_video, tpr_video, thresholds_video = roc_curve(labels, vid_preds_mean)
        auc_score_video = roc_auc_score(labels, vid_preds_mean)
        print(f"AUC (frame-level): {auc_score_video}")


    else:
        fpr, tpr, thresholds = roc_curve(labels, predictions)
        auc_score = roc_auc_score(labels, predictions)

        print(f"AUC (frame-level): {auc_score}")

        #Video-level AUC

        vids = {}

        for i, frame_str in enumerate(images):
            frame_path = Path(frame_str)
            video_path = str(frame_path.parent)
            
            if video_path in vids.keys():
                vids[video_path]["preds"].append(predictions[i])

            else:
                vids[video_path] = {
                    "preds": [predictions[i]],
                    "label": labels[i]
                }

        vids_mean = {}

        labels_videos_mean = []
        preds_videos_mean = []

        for video, preds_label in vids.items():
            video_mean = np.mean(preds_label["preds"])
            vids_mean[video] = {
            "pred": video_mean,
            "label": preds_label["label"]
                }
            
            labels_videos_mean.append(preds_label["label"])
            preds_videos_mean.append(video_mean)

        
        fpr_vid, tpr_vid, thresholds_vid = roc_curve(labels_videos_mean, preds_videos_mean)
        auc_score_video = roc_auc_score(labels_videos_mean, preds_videos_mean)

        print(f"AUC (video-level): {auc_score_video}")

    # Saving

    metrics = {
        "model_name": model_name,
        "test_dataset": test_dataset,
        "train_dataset": train_dataset,
        "frames_per_video_test": config["frame_num"]["test"],
        "clip_size": config["clip_size"] if model_name in vid_level_models else None,
        "elapsed_time": elapsed_time,
        "frame_auc": auc_score,
        "video_auc": auc_score_video
    }


    results = {
        "labels": labels,
        "preds": predictions,
        "paths": images
    }

    this_exp_path = f"{os.path.dirname(exps_dir)}/{config["test_dataset"][0]}/{model_name}/{train_dataset}"

    os.makedirs(this_exp_path, exist_ok=True)
    os.makedirs(f"{this_exp_path}/preds/video-level", exist_ok=True)

    with open(f"{this_exp_path}/{datetime.datetime.now(datetime.timezone(datetime.timedelta(hours=+1))).strftime("%Y-%m-%dT%H-%M-%SA")}.json", "x") as file:
        json.dump(metrics, file, cls=NumpyEncoder, indent=4)

    with open(f"{this_exp_path}/preds/{datetime.datetime.now(datetime.timezone(datetime.timedelta(hours=+1))).strftime("%Y-%m-%dT%H-%M-%SA")}.json", "x") as file:
        json.dump(results, file, cls=NumpyEncoder, indent=4)

    with open(f"{this_exp_path}/preds/video-level/{datetime.datetime.now(datetime.timezone(datetime.timedelta(hours=+1))).strftime("%Y-%m-%dT%H-%M-%SA")}.json", "x") as file:
        json.dump(vids_mean, file, cls=NumpyEncoder, indent=4)


# Entry Point

In [12]:
exps = pd.read_csv("exps.csv")

CONFIG_TEST = "config/test_config.yaml"
CONFIG_DETECTORS = "config/detector/"
WEIGHTS_BASE = "weights/"
EXPS_DIR = "exps/"

model_name = None
config_detector_path = None
weights_path = None
test_dataset = None

predictions_nps = None
label_nps = None
images = None
elapsed_time = None
config = None

for _, row in exps.iterrows():
    model_name = row['model_name']
    config_detector_path = row['config']
    weights_path = row ['weights']
    test_dataset = row['test_dataset']
    if row["done"] == True:
        print(f"Already done {model_name} ({weights_path}) on {test_dataset}. Skip")
        continue
    print(f"Start {model_name} ({weights_path} on {test_dataset})")

    test_data_loaders, model, device, config = init(CONFIG_DETECTORS + config_detector_path, WEIGHTS_BASE + weights_path, [test_dataset], CONFIG_TEST)

    start = time.time()

    predictions_nps, label_nps, images = test_epoch(model, test_data_loaders, device)

    stop = time.time()
    elapsed_time = stop-start
    save_display_metrics(predictions_nps, label_nps, images, model_name, test_dataset, weights_path, elapsed_time, config, EXPS_DIR)

    #if "i3d" in detector_name or "altfreezing" in detector_name:
    #    detector_results = get_prediction_df_video_level(predictions_nps, label_nps, images)
    #else:
    #detector_results = get_prediction_df(predictions_nps, label_nps, images)

    #write_results(detector_results, detector_config_path.split("/")[-1].replace(".yaml", ""), weights[index].split("/")[-1].replace(".pth", ""))

Begin spsl (train_on_df40-fs-ff/spsl.pth on FSAll_cdf)
LEN 1602 FSAll_Real
LEN 5161 FSAll_Fake
===> Load checkpoint done!


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 6688/6688 [07:25<00:00, 15.01it/s]


Results for spsl (train_on_df40-fs-ff) on FSAll_cdf (446.02345156669617s)
AUC (frame-level): 0.9376198737069196
AUC (video-level): 0.9736125957646916
Begin spsl (train_on_df40-fr-ff/spsl.pth on FSAll_cdf)
LEN 1602 FSAll_Real
LEN 5161 FSAll_Fake
===> Load checkpoint done!


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 6688/6688 [07:25<00:00, 15.00it/s]


Results for spsl (train_on_df40-fr-ff) on FSAll_cdf (446.2105076313019s)
AUC (frame-level): 0.4778770022345325
AUC (video-level): 0.49480709905100695
Begin srm (train_on_df40-fs-ff/srm.pth on FSAll_cdf)
LEN 1602 FSAll_Real
LEN 5161 FSAll_Fake


...

Results for i3d (train_on_df40-fs-ff) on FSAll_cdf (253.50070023536682s)
AUC (frame-level): 0.9672311034985586
AUC (frame-level): 0.9719382766898399
Begin i3d (train_on_df40-all-ff/i3d.pth on FSAll_cdf)
LEN 1602 FSAll_Real
Skipping video FSAll_Real_00170_inswap because it has less than clip_size (32) frames (31).
Skipping video FSAll_Real_00063_inswap because it has less than clip_size (32) frames (31).
Skipping video FSAll_Real_00213_inswap because it has less than clip_size (32) frames (30).
Skipping video FSAll_Real_00095_inswap because it has less than clip_size (32) frames (31).
Skipping video FSAll_Real_00106_inswap because it has less than clip_size (32) frames (31).
Skipping video FSAll_Real_00092_inswap because it has less than clip_size (32) frames (31).
Skipping video FSAll_Real_00168_inswap because it has less than clip_size (32) frames (31).
Skipping video FSAll_Real_00207_inswap because it has less than clip_size (32) frames (27).
Skipping video FSAll_Real_00253_inswap be

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1523/1523 [04:08<00:00,  6.14it/s]


Results for i3d (train_on_df40-all-ff) on FSAll_cdf (248.5892264842987s)
AUC (frame-level): 0.9043474864980023
AUC (frame-level): 0.913533558270893
Begin i3d (train_on_df40-fr-ff/i3d.pth on FSAll_cdf)
LEN 1602 FSAll_Real
Skipping video FSAll_Real_00170_inswap because it has less than clip_size (32) frames (31).
Skipping video FSAll_Real_00063_inswap because it has less than clip_size (32) frames (31).
Skipping video FSAll_Real_00213_inswap because it has less than clip_size (32) frames (30).
Skipping video FSAll_Real_00095_inswap because it has less than clip_size (32) frames (31).
Skipping video FSAll_Real_00106_inswap because it has less than clip_size (32) frames (31).
Skipping video FSAll_Real_00092_inswap because it has less than clip_size (32) frames (31).
Skipping video FSAll_Real_00168_inswap because it has less than clip_size (32) frames (31).
Skipping video FSAll_Real_00207_inswap because it has less than clip_size (32) frames (27).
Skipping video FSAll_Real_00253_inswap beca

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1523/1523 [04:08<00:00,  6.14it/s]


Results for i3d (train_on_df40-fr-ff) on FSAll_cdf (248.5999050140381s)
AUC (frame-level): 0.6306995220483477
AUC (frame-level): 0.6321834059318614
